In [ ]:
import nltk
import pandas as pd
import re
import joblib
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Download NLTK data
# -----------------------------
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# -----------------------------
# Load Data
# -----------------------------
train_df = pd.read_json("train.json")
test_df = pd.read_json("test.json")

# -----------------------------
# Dedup BEFORE preprocessing
# -----------------------------
train_df = train_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])
test_df = test_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])

# -----------------------------
# Preprocessing
# -----------------------------
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)

train_df["text"] = train_df["text"].apply(preprocess)
test_df["text"] = test_df["text"].apply(preprocess)

# -----------------------------
# Features / Labels
# -----------------------------
X_train = train_df["text"]
y_train = train_df["label"]
X_test = test_df["text"]
y_test = test_df["label"]

# -----------------------------
# Grid Search (expanded alpha range)
# -----------------------------
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
    analyzer="char_wb",  # character level instead of word
    ngram_range=(3, 5),
    sublinear_tf=True
)),
    ("classifier", CalibratedClassifierCV(LinearSVC()))  # CalibratedClassifierCV adds predict_proba support
])

param_grid = {
    "classifier__estimator__C": [0.01, 0.1, 1.0, 10.0, 11.0, 12.0]  # C = regularization
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_weighted"
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)

# -----------------------------
# Save model
# -----------------------------
joblib.dump(best_model, "sentiment_model_svm.joblib")
print("Model saved as sentiment_model.joblib")

# -----------------------------
# Predict on test dataset
# -----------------------------
test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)

# -----------------------------
# Evaluation
# -----------------------------
print("Accuracy:", accuracy_score(y_test, test_predictions))
print("\nClassification Report:\n", classification_report(y_test, test_predictions))

# -----------------------------
# Show some predictions
# -----------------------------
for i in range(5):
    text = test_df["text"].iloc[i]
    pred_label = "Positive" if test_predictions[i] == 1 else "Negative"
    print("Review:", text[:200], "...")
    print("Prediction:", pred_label)
    print("Positive Prob:", test_probabilities[i][1])
    print("Negative Prob:", test_probabilities[i][0])
    print("-" * 60)

# -----------------------------
# Load model (example)
# -----------------------------
# loaded_model = joblib.load("sentiment_model.joblib")
# loaded_model.predict(["some new review text"])